## 1. Grid Search

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# 1. 데이터 로드
train_df = pd.read_csv("train.csv")

# 2. 커스텀 전처리 함수 정의
def custom_preprocessing(df):
    df = df.copy()

    # 기본 결측치 처리
    df['title_sentiment_polarity'] = df['title_sentiment_polarity'].fillna(0.0)
    df['abs_title_sentiment_polarity'] = df['abs_title_sentiment_polarity'].fillna(0.0)
    df['n_tokens_title'] = df['n_tokens_title'].fillna(df['n_tokens_title'].median())
    df['n_tokens_content'] = df['n_tokens_content'].fillna(df['n_tokens_content'].median())
    df['num_imgs'] = df['num_imgs'].fillna(0)
    df['num_videos'] = df['num_videos'].fillna(0)

    # 키워드 계열 평균 대체
    kw_cols = [col for col in df.columns if col.startswith('kw_')]
    for col in kw_cols:
        df[col] = df[col].fillna(df[col].mean())

    # 로그 변환
    for col in ['self_reference_avg_sharess', 'num_hrefs', 'num_keywords']:
        df[col] = df[col].fillna(df[col].mean())
        df[col] = np.log1p(df[col])  # log(x + 1)

    # 범주형 처리 + 파생 변수
    df['data_channel'] = df['data_channel'].fillna('Other')
    df['weekday'] = df['weekday'].fillna(df['weekday'].mode()[0])
    df['is_weekend'] = df['weekday'].isin(['Saturday', 'Sunday']).astype(int)
    df = pd.get_dummies(df, columns=['data_channel', 'weekday'], prefix=['channel', 'day'])

    # 불필요한 열 제거
    df = df.drop(columns=['id', 'url', 'timedelta', 'shares'], errors='ignore')

    # 혹시 남은 NaN 제거
    df = df.dropna()

    return df

# 3. 전처리 적용
train_df_clean = custom_preprocessing(train_df)

# 4. 특성과 타겟 분리
X = train_df_clean.drop(columns=['y'])
y = train_df_clean['y']

# 5. 정규화 (StandardScaler)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 6. 학습/검증 분할
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 7. 하이퍼파라미터 그리드 정의
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 8, 10],
    'learning_rate': [0.05, 0.01, 0.001],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# 8. XGBoost + GridSearchCV
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
grid_search = GridSearchCV(xgb, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1) # scoring='f1'
grid_search.fit(X_train, y_train)

# 9. 최적 모델 평가
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_val)
y_prob = best_model.predict_proba(X_val)[:, 1]

acc = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
auc = roc_auc_score(y_val, y_prob)
final_score = (acc + f1 + auc) / 3

print("Best Parameters:", grid_search.best_params_)
print(f"Accuracy: {acc:.4f}, F1 Score: {f1:.4f}, AUC: {auc:.4f}")
print(f"Final Score (avg): {(final_score):.4f}")

Fitting 3 folds for each of 72 candidates, totalling 216 fits


KeyboardInterrupt: 

## 2. 전처리만 약간 다르게 함.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# 1. 데이터 로드
train_df = pd.read_csv("train.csv")

# 2. 정밀 전처리 함수
def strict_xgb_preprocessing(df):
    df = df.copy()

    # 수치형 → 중앙값
    numeric_fill_median = [
        'n_tokens_title', 'n_tokens_content', 'n_unique_tokens', 'n_non_stop_words',
        'n_non_stop_unique_tokens', 'num_hrefs', 'num_self_hrefs', 'num_imgs', 'num_videos'
    ]
    for col in numeric_fill_median:
        df[col] = df[col].fillna(df[col].median())

    # 감성 → 0.0 중립값
    df['title_sentiment_polarity'] = df['title_sentiment_polarity'].fillna(0.0)
    df['abs_title_sentiment_polarity'] = df['abs_title_sentiment_polarity'].fillna(0.0)

    # 로그 변환 변수들
    for col in ['self_reference_avg_sharess', 'num_keywords']:
        df[col] = df[col].fillna(df[col].mean())
        df[col] = np.log1p(df[col])  # log(x + 1)

    # 키워드 통계 → 평균 대체
    kw_cols = [col for col in df.columns if col.startswith('kw_')]
    for col in kw_cols:
        df[col] = df[col].fillna(df[col].mean())

    # 범주형 → 결측 대체 + 파생
    df['data_channel'] = df['data_channel'].fillna('Other')
    df['weekday'] = df['weekday'].fillna(df['weekday'].mode()[0])
    df['is_weekend'] = df['weekday'].isin(['Saturday', 'Sunday']).astype(int)

    # One-hot 인코딩
    df = pd.get_dummies(df, columns=['data_channel', 'weekday'], prefix=['channel', 'day'])

    # 불필요한 열 제거
    df = df.drop(columns=['id', 'url', 'timedelta', 'shares'], errors='ignore')

    # 최종 결측치 제거
    df = df.dropna()

    return df

# 3. 전처리 적용
train_df_clean = strict_xgb_preprocessing(train_df)

# 4. 특성과 타겟 분리
X = train_df_clean.drop(columns=['y'])
y = train_df_clean['y']

# 5. 정규화 (Standard Scaler)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 6. Train/Validation 분할
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 7. 하이퍼파라미터 그리드 설정
param_grid = {
    'n_estimators': [325, 330, 340],
    'max_depth': [9, 10],
    'learning_rate': [0.043, 0.04],
    'subsample': [0.7, 0.75],
    'colsample_bytree': [0.65, 0.7]
}

# 8. 모델 + GridSearchCV
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
grid_search = GridSearchCV(xgb, param_grid, cv=3, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

# 9. 최적 모델 평가
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_val)
y_prob = best_model.predict_proba(X_val)[:, 1]

acc = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
auc = roc_auc_score(y_val, y_prob)
final_score = (acc + f1 + auc) / 3

# 10. 결과 출력
print("Best Parameters:", grid_search.best_params_)
print(f"Accuracy: {acc:.4f}, F1 Score: {f1:.4f}, AUC: {auc:.4f}")
print(f"Final Score (avg): {final_score:.4f}")

Fitting 3 folds for each of 72 candidates, totalling 216 fits


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:50:55] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Best Parameters: {'colsample_bytree': 0.75, 'learning_rate': 0.04, 'max_depth': 9, 'n_estimators': 330, 'subsample': 0.7}
Accuracy: 0.6560, F1 Score: 0.6272, AUC: 0.7348
Final Score (avg): 0.6727


## 하이퍼파라미터 다르게

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# 데이터 로드
train_df = pd.read_csv("train.csv")

# 전처리 함수 (동일)
def strict_xgb_preprocessing(df):
    df = df.copy()
    numeric_fill_median = [
        'n_tokens_title', 'n_tokens_content', 'n_unique_tokens', 'n_non_stop_words',
        'n_non_stop_unique_tokens', 'num_hrefs', 'num_self_hrefs', 'num_imgs', 'num_videos'
    ]
    for col in numeric_fill_median:
        df[col] = df[col].fillna(df[col].median())

    df['title_sentiment_polarity'] = df['title_sentiment_polarity'].fillna(0.0)
    df['abs_title_sentiment_polarity'] = df['abs_title_sentiment_polarity'].fillna(0.0)

    for col in ['self_reference_avg_sharess', 'num_keywords']:
        df[col] = df[col].fillna(df[col].mean())
        df[col] = np.log1p(df[col])

    kw_cols = [col for col in df.columns if col.startswith('kw_')]
    for col in kw_cols:
        df[col] = df[col].fillna(df[col].mean())

    df['data_channel'] = df['data_channel'].fillna('Other')
    df['weekday'] = df['weekday'].fillna(df['weekday'].mode()[0])
    df['is_weekend'] = df['weekday'].isin(['Saturday', 'Sunday']).astype(int)

    df = pd.get_dummies(df, columns=['data_channel', 'weekday'], prefix=['channel', 'day'])

    df = df.drop(columns=['id', 'url', 'timedelta', 'shares'], errors='ignore')
    df = df.dropna()

    return df

# 전처리
train_df_clean = strict_xgb_preprocessing(train_df)
X = train_df_clean.drop(columns=['y'])
y = train_df_clean['y']
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 하이퍼파라미터 새 버전
param_grid = {
    'n_estimators': [320, 330, 335],
    'max_depth': [8, 9, 10],
    'learning_rate': [0.037, 0.04, 0.043],
    'subsample': [0.65, 0.75],
    'colsample_bytree': [0.6, 0.7]
}

# 모델 + 그리드 탐색
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
grid_search = GridSearchCV(xgb, param_grid, cv=3, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

# 예측 확률
y_prob = best_model.predict_proba(X_val)[:, 1]

# 여러 Threshold 테스트
thresholds = np.arange(0.45, 0.5, 0.55)
print("Best Params:", grid_search.best_params_)
print("Threshold Test Results:")
print("-" * 50)
for thresh in thresholds:
    y_pred_thresh = (y_prob >= thresh).astype(int)
    acc = accuracy_score(y_val, y_pred_thresh)
    f1 = f1_score(y_val, y_pred_thresh)
    auc = roc_auc_score(y_val, y_prob)
    final = (acc + f1 + auc) / 3
    print(f"Threshold: {thresh:.2f} | Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f} | Final Score: {final:.4f}")


Fitting 3 folds for each of 108 candidates, totalling 324 fits


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [13:19:30] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Best Params: {'colsample_bytree': 0.7, 'learning_rate': 0.037, 'max_depth': 8, 'n_estimators': 320, 'subsample': 0.65}
Threshold Test Results:
--------------------------------------------------
Threshold: 0.45 | Acc: 0.6355 | F1: 0.6279 | AUC: 0.7338 | Final Score: 0.6657


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# 데이터 로드
train_df = pd.read_csv("train.csv")

# 정밀 전처리
def strict_xgb_preprocessing(df):
    df = df.copy()
    numeric_fill_median = [
        'n_tokens_title', 'n_tokens_content', 'n_unique_tokens', 'n_non_stop_words',
        'n_non_stop_unique_tokens', 'num_hrefs', 'num_self_hrefs', 'num_imgs', 'num_videos'
    ]
    for col in numeric_fill_median:
        df[col] = df[col].fillna(df[col].median())

    df['title_sentiment_polarity'] = df['title_sentiment_polarity'].fillna(0.0)
    df['abs_title_sentiment_polarity'] = df['abs_title_sentiment_polarity'].fillna(0.0)

    for col in ['self_reference_avg_sharess', 'num_keywords']:
        df[col] = df[col].fillna(df[col].mean())
        df[col] = np.log1p(df[col])

    kw_cols = [col for col in df.columns if col.startswith('kw_')]
    for col in kw_cols:
        df[col] = df[col].fillna(df[col].mean())

    df['data_channel'] = df['data_channel'].fillna('Other')
    df['weekday'] = df['weekday'].fillna(df['weekday'].mode()[0])
    df['is_weekend'] = df['weekday'].isin(['Saturday', 'Sunday']).astype(int)

    df = pd.get_dummies(df, columns=['data_channel', 'weekday'], prefix=['channel', 'day'])

    df = df.drop(columns=['id', 'url', 'timedelta', 'shares'], errors='ignore')
    df = df.dropna()

    return df

# 전처리 적용
train_df_clean = strict_xgb_preprocessing(train_df)
X = train_df_clean.drop(columns=['y'])
y = train_df_clean['y']

# 정규화
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Train/Validation 분할
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 클래스 불균형 비율 계산
pos = sum(y_train == 1)
neg = sum(y_train == 0)
scale_ratio = neg / pos
print(f"[INFO] scale_pos_weight = {scale_ratio:.2f}")

# 하이퍼파라미터 그리드
param_grid = {
    'n_estimators': [320, 330, 340],
    'max_depth': [8, 9, 10],
    'learning_rate': [0.037, 0.04, 0.043],
    'subsample': [0.65, 0.75],
    'colsample_bytree': [0.6, 0.7]
}

# XGB 모델 정의 (간결하게)
xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=scale_ratio  # 클래스 불균형 대응
)

# 그리드 탐색
grid_search = GridSearchCV(
    xgb, param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

# 예측 확률
y_prob = best_model.predict_proba(X_val)[:, 1]

# Threshold 튜닝
print("\n[🔍 Threshold Test Results]")
print("-" * 55)
thresholds = np.arange(0.3, 0.71, 0.05)
best_score = 0
best_thresh = 0.5

for thresh in thresholds:
    y_pred = (y_prob >= thresh).astype(int)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    final = (acc + f1 + auc) / 3

    print(f"Threshold: {thresh:.2f} | Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f} | Final: {final:.4f}")

    if final > best_score:
        best_score = final
        best_thresh = thresh

# 최종 결과 출력
print("\n✅ Best Params:", grid_search.best_params_)
print(f"✅ Best Threshold: {best_thresh:.2f} | Final Score: {best_score:.4f}")

[INFO] scale_pos_weight = 1.00
Fitting 3 folds for each of 108 candidates, totalling 324 fits

[🔍 Threshold Test Results]
-------------------------------------------------------
Threshold: 0.30 | Acc: 0.6241 | F1: 0.6707 | AUC: 0.7321 | Final: 0.6756
Threshold: 0.35 | Acc: 0.6446 | F1: 0.6695 | AUC: 0.7321 | Final: 0.6821
Threshold: 0.40 | Acc: 0.6492 | F1: 0.6593 | AUC: 0.7321 | Final: 0.6802
Threshold: 0.45 | Acc: 0.6401 | F1: 0.6274 | AUC: 0.7321 | Final: 0.6665
Threshold: 0.50 | Acc: 0.6560 | F1: 0.6272 | AUC: 0.7321 | Final: 0.6718
Threshold: 0.55 | Acc: 0.6743 | F1: 0.6247 | AUC: 0.7321 | Final: 0.6770
Threshold: 0.60 | Acc: 0.6788 | F1: 0.6158 | AUC: 0.7321 | Final: 0.6756
Threshold: 0.65 | Acc: 0.6856 | F1: 0.6034 | AUC: 0.7321 | Final: 0.6737
Threshold: 0.70 | Acc: 0.6720 | F1: 0.5610 | AUC: 0.7321 | Final: 0.6550

✅ Best Params: {'colsample_bytree': 0.7, 'learning_rate': 0.043, 'max_depth': 10, 'n_estimators': 320, 'subsample': 0.65}
✅ Best Threshold: 0.35 | Final Score: 0.68

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# 데이터 로드
train_df = pd.read_csv("train.csv")

# 정밀 전처리
def strict_xgb_preprocessing(df):
    df = df.copy()
    numeric_fill_median = [
        'n_tokens_title', 'n_tokens_content', 'n_unique_tokens', 'n_non_stop_words',
        'n_non_stop_unique_tokens', 'num_hrefs', 'num_self_hrefs', 'num_imgs', 'num_videos'
    ]
    for col in numeric_fill_median:
        df[col] = df[col].fillna(df[col].median())

    df['title_sentiment_polarity'] = df['title_sentiment_polarity'].fillna(0.0)
    df['abs_title_sentiment_polarity'] = df['abs_title_sentiment_polarity'].fillna(0.0)

    for col in ['self_reference_avg_sharess', 'num_keywords']:
        df[col] = df[col].fillna(df[col].mean())
        df[col] = np.log1p(df[col])

    kw_cols = [col for col in df.columns if col.startswith('kw_')]
    for col in kw_cols:
        df[col] = df[col].fillna(df[col].mean())

    df['data_channel'] = df['data_channel'].fillna('Other')
    df['weekday'] = df['weekday'].fillna(df['weekday'].mode()[0])
    df['is_weekend'] = df['weekday'].isin(['Saturday', 'Sunday']).astype(int)

    df = pd.get_dummies(df, columns=['data_channel', 'weekday'], prefix=['channel', 'day'])

    df = df.drop(columns=['id', 'url', 'timedelta', 'shares'], errors='ignore')
    df = df.dropna()

    return df

# 전처리 적용
train_df_clean = strict_xgb_preprocessing(train_df)
X = train_df_clean.drop(columns=['y'])
y = train_df_clean['y']

# 정규화
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Train/Validation 분할
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 클래스 불균형 비율 계산
pos = sum(y_train == 1)
neg = sum(y_train == 0)
scale_ratio = neg / pos
print(f"[INFO] scale_pos_weight = {scale_ratio:.2f}")

# 하이퍼파라미터 그리드
param_grid = {
    'n_estimators': [310, 320, 330],
    'max_depth': [9, 10, 11],
    'learning_rate': [0.04, 0.043, 0.045],
    'subsample': [0.6, 0.65, 0.68],
    'colsample_bytree': [0.6, 0.7]
}

# XGB 모델 정의 (간결하게)
xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=scale_ratio  # ⚖️ 클래스 불균형 대응
)

# 그리드 탐색
grid_search = GridSearchCV(
    xgb, param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

# 예측 확률
y_prob = best_model.predict_proba(X_val)[:, 1]

# Threshold 튜닝
print("\n[🔍 Threshold Test Results]")
print("-" * 55)
thresholds = np.arange(0.3, 0.71, 0.05)
best_score = 0
best_thresh = 0.5

for thresh in thresholds:
    y_pred = (y_prob >= thresh).astype(int)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    final = (acc + f1 + auc) / 3

    print(f"Threshold: {thresh:.2f} | Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f} | Final: {final:.4f}")

    if final > best_score:
        best_score = final
        best_thresh = thresh

# 최종 결과 출력
print("\n✅ Best Params:", grid_search.best_params_)
print(f"✅ Best Threshold: {best_thresh:.2f} | Final Score: {best_score:.4f}")

[INFO] scale_pos_weight = 1.00
Fitting 3 folds for each of 162 candidates, totalling 486 fits

[🔍 Threshold Test Results]
-------------------------------------------------------
Threshold: 0.30 | Acc: 0.6241 | F1: 0.6707 | AUC: 0.7321 | Final: 0.6756
Threshold: 0.35 | Acc: 0.6446 | F1: 0.6695 | AUC: 0.7321 | Final: 0.6821
Threshold: 0.40 | Acc: 0.6492 | F1: 0.6593 | AUC: 0.7321 | Final: 0.6802
Threshold: 0.45 | Acc: 0.6401 | F1: 0.6274 | AUC: 0.7321 | Final: 0.6665
Threshold: 0.50 | Acc: 0.6560 | F1: 0.6272 | AUC: 0.7321 | Final: 0.6718
Threshold: 0.55 | Acc: 0.6743 | F1: 0.6247 | AUC: 0.7321 | Final: 0.6770
Threshold: 0.60 | Acc: 0.6788 | F1: 0.6158 | AUC: 0.7321 | Final: 0.6756
Threshold: 0.65 | Acc: 0.6856 | F1: 0.6034 | AUC: 0.7321 | Final: 0.6737
Threshold: 0.70 | Acc: 0.6720 | F1: 0.5610 | AUC: 0.7321 | Final: 0.6550

✅ Best Params: {'colsample_bytree': 0.7, 'learning_rate': 0.043, 'max_depth': 10, 'n_estimators': 320, 'subsample': 0.65}
✅ Best Threshold: 0.35 | Final Score: 0.68